# Traditional Machine Learning — 200k

This notebook trains TF-IDF Logistic Regression and Linear SVM models.

Results are in results/fixed_200k. Final test metrics and confusion matrices are calculated in notebook 10.

Settings:

- train / validation / test: 200,000 / 20,000 / 178,083
- TF-IDF: lowercase, sublinear TF, 1–2 grams, min_df 2, 50,000 features, float32
- Logistic Regression: C 1.0, liblinear, max_iter 1000, balanced classes
- Linear SVM: C 1.0, max_iter 5000, balanced classes, dual auto
- seed: 42

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = next(
    path for path in (Path.cwd(), Path.cwd().parent)
    if (path / "data/raw/train.csv").exists()
)

RESULTS_DIR = ROOT / "results/fixed_200k"
TRAIN_PATH = ROOT / "data/splits/200k/train.csv"
VALIDATION_PATH = ROOT / "data/splits/200k/validation.csv"
TEST_PATH = ROOT / "data/splits/full/test.csv"
TFIDF_SETTINGS = {
    "lowercase": True,
    "sublinear_tf": True,
    "ngram_range": (1, 2),
    "min_df": 2,
    "max_features": 50000,
    "dtype": np.float32,
}
MODEL_SETTINGS = {
    "logistic_regression": {"C": 1.0, "solver": "liblinear", "max_iter": 1000, "class_weight": "balanced", "random_state": 42},
    "linear_svm": {"C": 1.0, "max_iter": 5000, "class_weight": "balanced", "dual": "auto", "random_state": 42},
}

## Load and check the data

In [ ]:
train = pd.read_csv(TRAIN_PATH)
validation = pd.read_csv(VALIDATION_PATH)
test = pd.read_csv(TEST_PATH)
assert len(train) == 200000
assert len(validation) == 20000
assert len(test) == 178083


## Core TF-IDF and model training code

In [ ]:
import json
import time
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.svm import LinearSVC

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
pipeline_started = time.perf_counter()
vectorizer = TfidfVectorizer(**TFIDF_SETTINGS)
train_matrix = vectorizer.fit_transform(train["comment_text"])
validation_matrix = vectorizer.transform(validation["comment_text"])
test_matrix = vectorizer.transform(test["comment_text"])
tfidf_seconds = time.perf_counter() - pipeline_started
models = {
    "logistic_regression": LogisticRegression(**MODEL_SETTINGS["logistic_regression"]),
    "linear_svm": LinearSVC(**MODEL_SETTINGS["linear_svm"]),
}
for name, model in models.items():
    started = time.perf_counter()
    model.fit(train_matrix, train["label"])
    validation_prediction = model.predict(validation_matrix)
    training_seconds = time.perf_counter() - started
    prediction_started = time.perf_counter()
    test_prediction = model.predict(test_matrix)
    prediction_seconds = time.perf_counter() - prediction_started

    output_dir = RESULTS_DIR / name
    output_dir.mkdir(parents=True, exist_ok=True)
    prediction_output = test[["id", "comment_text", "target", "label"]].copy()
    prediction_output["predicted_label"] = test_prediction
    if name == "logistic_regression":
        prediction_output["toxic_probability"] = model.predict_proba(test_matrix)[:, 1]
    else:
        prediction_output["decision_score"] = model.decision_function(test_matrix)
    prediction_output.to_csv(output_dir / "predictions.csv", index=False)
    joblib.dump(model, output_dir / "model.joblib")
    joblib.dump(vectorizer, output_dir / "tfidf_vectorizer.joblib")
    (output_dir / "runtime.json").write_text(json.dumps({
        "tfidf_seconds": tfidf_seconds,
        "training_seconds": training_seconds,
        "prediction_seconds": prediction_seconds,
        "total_pipeline_seconds": tfidf_seconds + training_seconds + prediction_seconds,
    }, indent=2))
    print(name, "validation toxic F1:", f1_score(validation["label"], validation_prediction, zero_division=0))
    print(name, "training seconds:", round(training_seconds, 2))